# Feature engineering for Naive Bayes

Reuses the locked train/test split from `data/train_data.csv` and `data/test_data.csv`.

Compares four NB setups on macro F1:
1. TF-IDF(clean_content) — same as `baseline.ipynb`
2. TF-IDF(spaCy lemmas of raw content)
3. TF-IDF(clean_content) + POS-tag distribution per doc
4. TF-IDF(clean_content) + LDA topic distribution per doc

Each setup fits its own TfidfVectorizer on the training fold only — no leakage from the test set.

In [9]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import spacy
from scipy.sparse import hstack, csr_matrix
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import f1_score
from sklearn.naive_bayes import MultinomialNB

cols = ['appName', 'content', 'clean_content', 'Sentiment']
train_df = pd.read_csv('../data/train_data.csv')[cols].reset_index(drop=True)
test_df = pd.read_csv('../data/test_data.csv')[cols].reset_index(drop=True)

y_train = train_df['Sentiment']
y_test = test_df['Sentiment']
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

Train: 16418 | Test: 4105


## 1. Baseline reference: NB on clean_content

In [10]:
tfidf = TfidfVectorizer(max_features=5000)
X_tr = tfidf.fit_transform(train_df['clean_content'])
X_te = tfidf.transform(test_df['clean_content'])

nb_clean = MultinomialNB().fit(X_tr, y_train)
f1_clean = f1_score(y_test, nb_clean.predict(X_te), average='macro')
print(f'NB on clean_content: macro F1 = {f1_clean:.4f}')

NB on clean_content: macro F1 = 0.5742


## 2. Lemmatization + POS extraction (single spaCy pass)

Runs spaCy once over the RAW content of train+test, disabling parser/NER for speed. Produces both lemmatized strings and per-doc POS-tag distributions so the feature experiments below don't each re-tokenize.

In [11]:
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

POS_TAGS = ['NOUN', 'VERB', 'ADJ', 'ADV', 'PROPN', 'PRON', 'AUX', 'INTJ', 'DET', 'ADP']
POS_IDX = {t: i for i, t in enumerate(POS_TAGS)}

def spacy_features(texts):
    """One spaCy pass. Returns (lemma_strings, pos_distribution_matrix)."""
    lemmas = []
    pos = np.zeros((len(texts), len(POS_TAGS)), dtype=np.float32)
    for i, doc in enumerate(nlp.pipe(texts, batch_size=256)):
        toks = [t for t in doc if not (t.is_space or t.is_punct)]
        lemmas.append(' '.join(t.lemma_.lower() for t in toks if t.lemma_.strip()))
        n = len(toks) or 1
        for t in toks:
            j = POS_IDX.get(t.pos_)
            if j is not None:
                pos[i, j] += 1.0
        pos[i] /= n
    return lemmas, pos

print('Processing train...')
train_lemmas, pos_train = spacy_features(train_df['content'].astype(str).tolist())
print('Processing test...')
test_lemmas, pos_test = spacy_features(test_df['content'].astype(str).tolist())
print('Done')

Processing train...
Processing test...
Done


## 3. NB on lemmatized text

In [12]:
tfidf_l = TfidfVectorizer(max_features=5000)
X_tr_l = tfidf_l.fit_transform(train_lemmas)
X_te_l = tfidf_l.transform(test_lemmas)

nb_lemma = MultinomialNB().fit(X_tr_l, y_train)
f1_lemma = f1_score(y_test, nb_lemma.predict(X_te_l), average='macro')
print(f'NB on lemmatized content: macro F1 = {f1_lemma:.4f}')

NB on lemmatized content: macro F1 = 0.5752


## 4. NB on TF-IDF + POS-tag distribution

POS distributions are per-doc frequencies across the ten tags in `POS_TAGS`, all in [0, 1] — non-negative, so MultinomialNB accepts them alongside TF-IDF.

In [13]:
X_tr_pos = hstack([X_tr, csr_matrix(pos_train)])
X_te_pos = hstack([X_te, csr_matrix(pos_test)])

nb_pos = MultinomialNB().fit(X_tr_pos, y_train)
f1_pos = f1_score(y_test, nb_pos.predict(X_te_pos), average='macro')
print(f'NB on TF-IDF + POS distribution: macro F1 = {f1_pos:.4f}')

NB on TF-IDF + POS distribution: macro F1 = 0.5719


## 5. NB on TF-IDF + LDA topic distribution

Fits LDA on the training fold only, then transforms both train and test to topic-probability vectors and concatenates them to the TF-IDF matrix.

In [14]:
N_TOPICS = 10

cv = CountVectorizer(max_features=5000, stop_words='english')
C_tr = cv.fit_transform(train_df['clean_content'])
C_te = cv.transform(test_df['clean_content'])

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    learning_method='online',
)
lda.fit(C_tr)
lda_tr = lda.transform(C_tr)
lda_te = lda.transform(C_te)

X_tr_lda = hstack([X_tr, csr_matrix(lda_tr)])
X_te_lda = hstack([X_te, csr_matrix(lda_te)])

nb_lda = MultinomialNB().fit(X_tr_lda, y_train)
f1_lda = f1_score(y_test, nb_lda.predict(X_te_lda), average='macro')
print(f'NB on TF-IDF + LDA({N_TOPICS} topics): macro F1 = {f1_lda:.4f}')

NB on TF-IDF + LDA(10 topics): macro F1 = 0.5714


## 6. LDA topics vs. sentiment

Top words per topic, then the mean topic weight for each sentiment class on the training set — shows which topics lean Positive, Negative, or Neutral.

In [15]:
feature_names = cv.get_feature_names_out()
print('Top words per LDA topic:')
for k, topic in enumerate(lda.components_):
    top = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f'  Topic {k:>2}: {" ".join(top)}')

topic_cols = [f'T{i}' for i in range(N_TOPICS)]
topic_df = pd.DataFrame(lda_tr, columns=topic_cols)
topic_df['Sentiment'] = y_train.values
print('\nMean topic weight per sentiment class (train):')
print(topic_df.groupby('Sentiment').mean().round(3))

Top words per LDA topic:
  Topic  0: need app image help fix images like chats create time
  Topic  1: good app dont bad helpful phone use experience like super
  Topic  2: gemini like use google chatgpt love ai update version assistant
  Topic  3: ai best app answer want worst dont problem questions slow
  Topic  4: time free limit message use hours information messages makes send
  Topic  5: chat gpt new voice response didnt annoying stars responses better
  Topic  6: nice hai app fast thank bahut option application wonderful bhi
  Topic  7: app useful im using times thats tried phone going says
  Topic  8: really claude great better make ask pro amazing subscription keeps
  Topic  9: work working answers things wrong excellent doesnt ok star properly

Mean topic weight per sentiment class (train):
              T0     T1     T2     T3     T4     T5     T6     T7     T8  \
Sentiment                                                                  
Negative   0.120  0.127  0.115  0.10

## 7. Summary

In [16]:
summary = pd.DataFrame([
    {'setup': 'TF-IDF(clean_content)',           'macro_f1': round(f1_clean, 4)},
    {'setup': 'TF-IDF(lemmas)',                  'macro_f1': round(f1_lemma, 4)},
    {'setup': 'TF-IDF(clean_content) + POS',     'macro_f1': round(f1_pos, 4)},
    {'setup': f'TF-IDF(clean_content) + LDA{N_TOPICS}', 'macro_f1': round(f1_lda, 4)},
])
print(summary.to_string(index=False))

                        setup  macro_f1
        TF-IDF(clean_content)    0.5742
               TF-IDF(lemmas)    0.5752
  TF-IDF(clean_content) + POS    0.5719
TF-IDF(clean_content) + LDA10    0.5714
